In [ ]:
import pandas as pd
import plotly.express as px

import importlib
import sys

if "snakemake" in locals():
    spec = importlib.util.spec_from_file_location("lib", "lib/__init__.py")
else:
    spec = importlib.util.spec_from_file_location("lib", "../lib/__init__.py")
module_obj = importlib.util.module_from_spec(spec)
sys.modules["lib"] = module_obj
spec.loader.exec_module(module_obj)

from lib import PipelineConfig

In [ ]:
if "snakemake" in locals():
    pipeline_config = PipelineConfig(snakemake.params.config, **snakemake.params.pipeline_kwargs)
    analysis_name = snakemake.params.analysis_name
    analysis = pipeline_config.analyses[analysis_name]
    assert analysis.type == "routing"

    scenarios = snakemake.params.scenarios
    reference = analysis.reference
    labels = {scenario_id: item[1] for scenario_id, item in analysis.scenario_items.items()}

    assert set(scenarios.keys()) == set(analysis.scenario_items.keys())
else: # We use simulations included in the base config for testing purposes
    scenarios = dict(nofeeder="../outputs/routing/outputs/simulated_nofeeder_pt",
                     small_area_feeder="../outputs/routing/outputs/simulated_feeder_transitWithAbstractAccess_200_300_0",
                     wide_area_feeder = "../outputs/routing/outputs/simulated_feeder_transitWithAbstractAccess_400_300_0")
    reference = "nofeeder"
    labels = {key: key for key in scenarios}

# Some sanity checks on the parameters
assert isinstance(scenarios, dict) and reference in scenarios
assert set(scenarios.keys()) == set(labels.keys())

In [ ]:
# We load the trips
dfs_trips = []
for s in scenarios:
    df_trips = pd.read_csv("%s/eqasim_trips.csv" % scenarios[s], sep=";")
    df_trips["scenario"] = s
    dfs_trips.append(df_trips)

df_trips = pd.concat(dfs_trips)
dfs_trips = [] # Just the free the memory behind the individual dfs

# Travel time comparison
Below we simply compare the travel times for the trips to give a example of code

In [ ]:
df_comparison = df_trips.pivot_table(values="travel_time", columns="scenario", index=["person_id", "person_trip_id"]).reset_index()

In [ ]:
df_plot = df_comparison.melt(id_vars=["person_id", "person_trip_id"], value_name="travel_time", var_name="scenario").sort_values(["person_id", "person_trip_id", "scenario"])

df_plot["scenario"] = df_plot["scenario"].map(labels)
fig = px.histogram(df_plot, x="travel_time", color="scenario", barmode="overlay", title="Travel time comparison")
fig.show()

In [ ]:
for c in df_comparison.columns:
    if c in ["person_id", "person_trip_id", reference]:
        continue
    df_comparison["gain_%s" % c]  = df_comparison[reference] - df_comparison[c]

df_plot = df_comparison.melt(id_vars=["person_id", "person_trip_id"], value_vars=[c for c in df_comparison.columns if c.startswith("gain_")], value_name="gain", var_name="scenario")

df_plot["scenario"] = df_plot["scenario"].apply(lambda s: labels[s[5::]])

fig = px.histogram(df_plot[df_plot["gain"] != 0], x="gain", color="scenario", barmode="overlay")
fig.show()

# What to do next ?

Same thing as for the routing based analysis of feeder services, we compute the route costs of the trips using their components. The only difference is that here we will mainly use bar plots with the scenario on the x axis (or color) instead of line plots with the feeder radius on the x axis.

At this step it might be worth it to start factoring some code in some separate python files and importing it where needed. I am thinking about the code that computes the routing cost components for PT trips which is also used in the notebook for the feeder sensitivity analysis.

Feel free to propose other plots.